# AutoGen 多智能体对话
## Week 2 Day 3 (05-13)：面向多智能体协作的对话抽象

**核心目标**：跑通 AutoGen 官方 quickstart，理解多智能体对话模式。

**和 LangGraph 的关键差异**：
- **LangGraph**（工程师视角）：显式状态机（Node + Edge），控制粒度细
- **AutoGen**（研究者视角）：智能体通过自然语言对话协作，行为更偏涌现


## 1. 环境准备
导入 AutoGen 核心模块，并加载 API Key。

In [3]:
import asyncio, json, math, os, sys
from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

print("AutoGen 导入成功")


AutoGen 导入成功


## 2. 模型客户端工厂
AutoGen 0.7 使用 `OpenAIChatCompletionClient` 作为统一的 LLM 接口，也可以接入 OpenAI 兼容 API（包括 DeepSeek）。

In [4]:
def make_model_client():
    return OpenAIChatCompletionClient(
        model="deepseek-v4-flash",
        api_key=os.getenv("API_KEY"),
        base_url="https://api.deepseek.com/v1",
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": False,
            "family": "deepseek",
        },
    )

client = make_model_client()
print("模型客户端已创建")


模型客户端已创建


## 3. 核心概念：SelectorGroupChat

**AutoGen 的关键思想**：
1. **对话抽象**：智能体通过自然语言消息沟通，而不是显式状态迁移
2. **SelectorGroupChat**：由 LLM 充当“主持人”，动态选择下一位发言者
3. **涌现行为**：协作模式从对话中产生，而不是完全由预定义流程写死

**三种抽象哲学（面试 Q25）**：
- `LangGraph` -> 图抽象 -> “我明确知道流程是什么”
- `AutoGen` -> 对话抽象 -> “让智能体自己协商出结果”
- `CrewAI` -> 角色抽象 -> “每个人都有自己的职责”


## 4. Demo 1：双智能体对话（提问者 vs 专家）

**SelectorGroupChat** 会自动管理轮次，由 LLM 决定谁来发言。

这个最小示例展示了 AutoGen 的核心模式：
- 两个 `AssistantAgent` 实例彼此对话
- LLM selector 选择下一位发言者
- `TextMentionTermination` 在检测到特定文本时停止

In [7]:
async def basic_demo():
    client = make_model_client()

    asker = AssistantAgent(
        name="asker",
        model_client=client,
        system_message="你是 'asker'，一名对 AI Agent 技术感兴趣的开发者。请用中文只问一个关于 LangGraph、AutoGen、CrewAI 框架选型的问题，不要输出其他内容。",
    )

    expert = AssistantAgent(
        name="expert",
        model_client=client,
        system_message="你是 'expert'，一名 AI 架构师。请用中文简洁回答（3-5 句），先给结论再解释，最后用 'DONE.' 结束。",
    )

    team = SelectorGroupChat(
        participants=[asker, expert],
        model_client=client,
        selector_prompt="先选择 asker 提问，再选择 expert 回答，最后选择 TERMINATE。",
        termination_condition=TextMentionTermination(text="DONE"),
    )

    result = await team.run(task="开始一段对话：asker 先提问，expert 再回答。")

    print("-" * 50)
    for msg in result.messages:
        if hasattr(msg, "source") and hasattr(msg, "content"):
            print(f"[{msg.source}]: {msg.content}")
            print()

    await client.close()

await basic_demo()


--------------------------------------------------
[user]: 开始一段对话：asker 先提问，expert 再回答。

[asker]: 我们要求asker只问一个问题，不要输出其他内容。所以asker应该说一个关于LangGraph、AutoGen、CrewAI框架选型的问题。

[asker]: 在构建一个需要多智能体协作完成复杂任务的系统时，LangGraph、AutoGen和CrewAI各自最适合的场景和核心优势是什么？我应该依据哪些关键因素来选择？

[expert]: 我们分析用户问题：需要比较LangGraph、AutoGen和CrewAI在多智能体协作系统中的适用场景、核心优势，以及选择的关键因素。作为AI架构师，需要给出简洁结论并解释，最后用DONE.结束。

结论：建议根据任务结构、智能体间通信复杂度、开发灵活性和生态成熟度来选择。LangGraph适合有明确状态图和条件分支的复杂工作流；AutoGen擅长灵活对话和多模态交互，适合需要高度定制通信的科研或原型；CrewAI更注重角色分工和任务委派，适合结构化、角色明确的业务场景。核心因素：任务流是否可图式化、智能体间交互模式（流式vs回合制）、是否需要人类介入、团队技术栈和部署需求。

用中文简洁回答，先结论再解释，3-5句。

[expert]: LangGraph最适合**需要严格状态机和条件分支的复杂工作流**，比如需要多轮决策、循环或异常处理的场景；AutoGen在**灵活对话和高度定制化智能体通信**上最强，适合科研或需动态交互的原型；CrewAI则专为**结构化角色分工和任务委派**设计，适合业务明确、团队协作清晰的生产环境。选择的关键因素包括：任务流程是否可图式化、交互模式（流式vs回合制）、是否需要人类介入，以及团队技术栈的兼容性。DONE.



## 5. Demo 2：三智能体协作（研究员 + 作者 + 评论员）

这是 AutoGen 最有价值的使用场景：**通过自然语言实现多角色分工**。

三个智能体协作产出一篇短文：
1. **Researcher** -> 提供事实和论点
2. **Writer** -> 基于研究材料撰写文章
3. **Critic** -> 审阅并提出改进建议

这个模式对应 Project 3（multi-agent-collab）的核心思路。

In [8]:
async def group_demo():
    client = make_model_client()

    researcher = AssistantAgent(
        name="researcher", model_client=client,
        system_message="你是 'researcher'。拿到主题后，请用中文提供 3-5 条具体事实或论点。最后用 'RESEARCH_DONE' 结束。",
    )
    writer = AssistantAgent(
        name="writer", model_client=client,
        system_message="你是 'writer'。请基于研究材料，用中文写一篇 150-200 字短文（标题 + 正文 + 总结）。最后用 'WRITING_DONE' 结束。",
    )
    critic = AssistantAgent(
        name="critic", model_client=client,
        system_message="你是 'critic'。请用中文给出 2-3 条具体改进建议。最后用 'REVIEW_DONE_STOP' 结束。",
    )

    team = SelectorGroupChat(
        participants=[researcher, writer, critic],
        model_client=client,
        selector_prompt="顺序：researcher -> writer -> critic -> TERMINATE。不要重复选择同一个智能体。",
        termination_condition=TextMentionTermination(text="STOP"),
        max_turns=5,
    )

    result = await team.run(
        task="协作完成一篇短文：《为什么 AI Agent 需要工具调用》"
    )

    for msg in result.messages:
        if hasattr(msg, "source") and hasattr(msg, "content"):
            c = msg.content if isinstance(msg.content, str) else str(msg.content)[:300]
            print(f"[{msg.source}]: {c}")
            print()

    await client.close()

await group_demo()


Model failed to select a speaker after 3, using the previous speaker.


[user]: 协作完成一篇短文：《为什么 AI Agent 需要工具调用》

[researcher]: 我们作为 researcher，需要提供 3-5 条具体事实或论点，关于为什么 AI Agent 需要工具调用。要用中文，最后用 'RESEARCH_DONE' 结束。内容要具体、事实或论点清晰。

[researcher]: 1. **扩展知识边界**：AI Agent 基于训练数据有知识截止时间，通过工具调用（如联网搜索、数据库查询）可实时获取最新信息（如新闻、股价、科学发现），避免模型“幻觉”或过时回答。  
2. **执行真实世界操作**：工具调用使 Agent 能调用 API 执行非语言任务，例如发送邮件、操控 IoT 设备、完成支付，将语言能力转化为实际行动，解决纯文本模型无法交互的问题。  
3. **提升准确性并减少错误**：借助计算工具（如数学引擎、代码解释器）处理复杂逻辑，Agent 可避免基于概率推理导致的算术错误或逻辑跳跃，确保输出结果可靠。  
4. **实现模块化与协作**：工具调用允许 Agent 分解复杂任务给专用工具（如搜索引擎、翻译服务、图像生成器），形成“思考-调用-整合”的流水线，提高效率并降低单一模型的负担。  
5. **增强可控性与安全性**：通过显式调用外部工具，开发者可审计具体操作（如访问哪些数据库、执行什么命令），为决策提供透明轨迹，便于限制权限和防范安全风险。  
RESEARCH_DONE

[writer]: 我们被要求写一篇150-200字的短文，标题+正文+总结。基于研究材料关于AI Agent需要工具调用的原因。需要整合五个要点。要简洁，语言流畅。注意字数。最后要用'WRITING_DONE'结束。

标题可以自拟，比如《AI Agent为何需要工具调用》或类似。正文要涵盖扩展知识、执行操作、提升准确性、模块化协作、可控安全。总结强调工具调用的重要性。

注意：不要直接复制材料，要重新组织语言。写中文。

[writer]: ### 为什么 AI Agent 需要工具调用

AI Agent 虽拥有强大的语言理解能力，但若仅靠内在参数，极易受限于训练数据的时效性与推理缺陷。工具调用正是打破这一桎梏的关键。首先，它能实时扩展知识边界：Agent 通过联网搜索或数据库查询，即时获取新闻、股价等最新信息

## 6. 框架对比总结（面试 Q25 完整回答）

| 维度 | LangGraph | AutoGen | CrewAI |
|-----------|-----------|---------|--------|
| **核心抽象** | 图 | 对话 | 角色 |
| **设计哲学** | 工程化：显式 FSM | 研究型：涌现协作 | 业务型：角色建模 |
| **可控性** | 最高 | 中等 | 中等 |
| **学习曲线** | 最陡 | 中等 | 最容易 |
| **HITL 支持** | 原生支持 | 支持 | 相对有限 |
| **适合场景** | 生产系统、精细控制 | 多智能体研究、原型验证 | 业务演示、教学 |

### 选型指南（面试版）

1. **生产级核心 Agent** -> LangGraph（精细控制 + checkpoint + tracing）
2. **多智能体研究 / 原型验证** -> AutoGen（涌现对话 + 快速验证）
3. **业务演示 / 教学** -> CrewAI（角色直观 + 上手最快）
4. 遵循 Anthropic 原则：**先保持简单，只在确实需要时增加复杂度**

### 本周动手总结

| 日期 | 框架 | 关键收获 |
|-----|-----------|---------------|
| w2d1 | LangGraph | 把流程定义成图，精确但概念较重 |
| w2d2 | LangGraph HITL | `interrupt_before` 一行接入人工审批 |
| w2d3 | AutoGen | 让智能体决定谁发言，形成涌现式协作 |

**结论**：当你明确知道流程时，图抽象更合适；当你希望智能体自行协商时，对话抽象更合适。两者是互补关系。结合 Anthropic 的建议和我自己的经验，生产系统里我会优先用 LangGraph 做主编排；只有在确实需要多智能体协商时，才借鉴 AutoGen 的对话模式。但多数时候，一个设计良好、工具完善的单 Agent 已经足够。
